# Hybrid Recommender

**Run ID:** HYBRID_RUN_001

This notebook develops the hybrid recommendation component for intelligent task allocation.

The hybrid recommender combines:

1. Content-Based Filtering
2. Collaborative Filtering
3. Knowledge Graph reasoning

The objective is to combine the strengths of the three individual recommendation approaches into a single ranking score.

The model is developed and tuned using the training and validation datasets only. The test dataset remains reserved for final evaluation.

The current experiment uses synthetic development data while awaiting the approved industry dataset. Results therefore represent development evidence and not final real-world performance.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import json

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

data_raw_path = project_root / "data" / "raw"
data_processed_path = project_root / "data" / "processed"
experiments_path = project_root / "experiments"

print("Project root:", project_root)

Project root: c:\Users\sagoo\OneDrive - Strathmore University\YEAR 4\SEM 1\IS Project II\intelligent-task-allocation


## 1. Load Development Data

The hybrid recommender uses the same development dataset and data split used by the individual recommendation models.

Training interactions are used to construct historical recommendation features.

Validation interactions are used for model development and weight selection.

The test dataset remains untouched.

In [3]:
agents_df = pd.read_csv(
    data_raw_path / "agents.csv"
)

skills_df = pd.read_csv(
    data_raw_path / "skills.csv"
)

agent_skills_df = pd.read_csv(
    data_raw_path / "agent_skills.csv"
)

tasks_df = pd.read_csv(
    data_raw_path / "tasks.csv"
)

task_skills_df = pd.read_csv(
    data_raw_path / "task_skills.csv"
)

train_df = pd.read_csv(
    data_processed_path / "train_interactions.csv"
)

validation_df = pd.read_csv(
    data_processed_path / "validation_interactions.csv"
)

test_df = pd.read_csv(
    data_processed_path / "test_interactions.csv"
)

print("Agents:", len(agents_df))
print("Tasks:", len(tasks_df))
print("Training interactions:", len(train_df))
print("Validation interactions:", len(validation_df))
print("Test interactions:", len(test_df))

Agents: 100
Tasks: 2000
Training interactions: 912
Validation interactions: 196
Test interactions: 196


In [4]:
cbf_metrics = pd.read_csv(
    experiments_path
    / "CBF_RUN_001"
    / "validation_metrics.csv"
)

cf_metrics = pd.read_csv(
    experiments_path
    / "CF_RUN_001"
    / "validation_metrics.csv"
)

kg_metrics = pd.read_csv(
    experiments_path
    / "KG_RUN_001"
    / "validation_metrics.csv"
)

print("Content-Based")
display(cbf_metrics)

print("Collaborative Filtering")
display(cf_metrics)

print("Knowledge Graph")
display(kg_metrics)

Content-Based


,K,Precision@K,Recall@K,Hit Rate@K,MAP@K
0,1,0.091837,0.091837,0.091837,0.091837
1,3,0.069728,0.209184,0.209184,0.138605
2,5,0.051020,0.255102,0.255102,0.149575


Collaborative Filtering


,K,Precision@K,Recall@K,Hit Rate@K,MAP@K
0,1,0.015306,0.015306,0.015306,0.015306
1,3,0.017007,0.051020,0.051020,0.030612
2,5,0.018367,0.091837,0.091837,0.040051


Knowledge Graph


,K,Precision@K,Recall@K,Hit Rate@K,MAP@K
0,1,0.091837,0.091837,0.091837,0.091837
1,3,0.076531,0.229592,0.229592,0.152211
2,5,0.068367,0.341837,0.341837,0.176956


## 2. Rebuild Individual Recommendation Components

The hybrid recommender requires a score from each of the three recommendation approaches for every candidate agent.

The individual scoring functions are therefore reconstructed in this notebook using the same logic used in the previous experiments.

Only training data is used for historical information.

### 2.1 Content-Based Component

In [5]:
proficiency_map = {
    "Beginner": 1,
    "Intermediate": 2,
    "Advanced": 3
}

CBF_WEIGHTS = {
    "skill_match": 0.45,
    "specialization_match": 0.20,
    "location_match": 0.10,
    "performance": 0.10,
    "workload": 0.10,
    "experience": 0.05
}

print("CBF weight total:", sum(CBF_WEIGHTS.values()))

CBF weight total: 1.0


In [6]:
agents_lookup = (
    agents_df
    .set_index("agent_id")
    .to_dict("index")
)

tasks_lookup = (
    tasks_df
    .set_index("task_id")
    .to_dict("index")
)


agent_skill_lookup = {}

for _, row in agent_skills_df.iterrows():

    agent_skill_lookup[
        (row["agent_id"], row["skill_id"])
    ] = proficiency_map[
        row["proficiency_level"]
    ]


task_skill_lookup = {}

for _, row in task_skills_df.iterrows():

    task_id = row["task_id"]

    if task_id not in task_skill_lookup:
        task_skill_lookup[task_id] = []

    task_skill_lookup[task_id].append(
        (
            row["skill_id"],
            proficiency_map[
                row["required_level"]
            ]
        )
    )


max_workload = agents_df[
    "workload"
].max()

max_experience = agents_df[
    "experience_years"
].max()


print(
    "Agents indexed:",
    len(agents_lookup)
)

print(
    "Tasks indexed:",
    len(tasks_lookup)
)

Agents indexed: 100
Tasks indexed: 2000


In [7]:
def calculate_cbf_score(
    agent_id,
    task_id
):

    agent = agents_lookup[agent_id]
    task = tasks_lookup[task_id]

    # ---------------------------------------
    # Skill match
    # ---------------------------------------

    required_skills = task_skill_lookup.get(
        task_id,
        []
    )

    skill_scores = []

    for skill_id, required_level in required_skills:

        agent_level = agent_skill_lookup.get(
            (agent_id, skill_id),
            0
        )

        if agent_level == 0:
            skill_scores.append(0.0)

        else:
            skill_scores.append(
                min(
                    agent_level / required_level,
                    1.0
                )
            )

    if skill_scores:
        skill_match = np.mean(skill_scores)
    else:
        skill_match = 0.0


    # ---------------------------------------
    # Specialization
    # ---------------------------------------

    specialization_match = (
        1.0
        if str(agent["specialization"]).strip().lower()
        == str(task["task_type"]).strip().lower()
        else 0.0
    )


    # ---------------------------------------
    # Location
    # ---------------------------------------

    location_match = (
        1.0
        if str(agent["location"]).strip().lower()
        == str(task["location"]).strip().lower()
        else 0.0
    )


    # ---------------------------------------
    # Performance
    # ---------------------------------------

    performance_score = (
        float(agent["performance_score"])
        / 100.0
    )


    # ---------------------------------------
    # Workload
    # ---------------------------------------

    if max_workload > 0:

        workload_score = (
            1.0
            - (
                float(agent["workload"])
                / max_workload
            )
        )

    else:
        workload_score = 1.0


    # ---------------------------------------
    # Experience
    # ---------------------------------------

    if max_experience > 0:

        experience_score = (
            float(agent["experience_years"])
            / max_experience
        )

    else:
        experience_score = 0.0


    # ---------------------------------------
    # Final CBF score
    # ---------------------------------------

    cbf_score = (
        CBF_WEIGHTS["skill_match"]
        * skill_match

        + CBF_WEIGHTS["specialization_match"]
        * specialization_match

        + CBF_WEIGHTS["location_match"]
        * location_match

        + CBF_WEIGHTS["performance"]
        * performance_score

        + CBF_WEIGHTS["workload"]
        * workload_score

        + CBF_WEIGHTS["experience"]
        * experience_score
    )


    return {
        "cbf_score": float(cbf_score),
        "skill_match": float(skill_match),
        "specialization_match":
            float(specialization_match),
        "location_match":
            float(location_match)
    }

In [8]:
sample_validation = validation_df.iloc[0]

sample_task_id = sample_validation["task_id"]
sample_agent_id = sample_validation["agent_id"]

print("Task:", sample_task_id)
print("Agent:", sample_agent_id)

calculate_cbf_score(
    sample_agent_id,
    sample_task_id
)

Task: T1892
Agent: A058


{'cbf_score': 0.6961600000000001,
 'skill_match': 1.0,
 'specialization_match': 0.0,
 'location_match': 1.0}

### 2.2 Collaborative Filtering Component

Collaborative Filtering uses historical training interactions to identify agents with similar task-performance patterns.

Because individual tasks are sparse, historical ratings are aggregated by task type. Agent similarity is then calculated using cosine similarity.

The predicted performance rating is normalized to a 0–1 score so that it can later be combined with the Content-Based and Knowledge Graph scores.

In [10]:
# Start with a copy of the training data
train_cf_df = train_df.copy()

# Only add task_type if it is not already present
if "task_type" not in train_cf_df.columns:

    train_cf_df = train_cf_df.merge(
        tasks_df[["task_id", "task_type"]],
        on="task_id",
        how="left"
    )


# Confirm columns
print("Columns:", train_cf_df.columns.tolist())


# Create Agent × Task Type rating matrix
cf_rating_matrix = train_cf_df.pivot_table(
    index="agent_id",
    columns="task_type",
    values="performance_rating",
    aggfunc="mean"
)


# Ensure all agents are represented
cf_rating_matrix = cf_rating_matrix.reindex(
    agents_df["agent_id"]
)


# Keep missing values for prediction
cf_rating_matrix_raw = cf_rating_matrix.copy()


# Fill missing values only for similarity calculation
cf_rating_matrix_filled = (
    cf_rating_matrix.fillna(0)
)


print(
    "CF matrix shape:",
    cf_rating_matrix.shape
)

display(
    cf_rating_matrix.head()
)

Columns: ['assignment_id', 'task_id', 'agent_id', 'status', 'assigned_at', 'completed_at', 'completion_time_hours', 'performance_rating', 'task_type', 'location', 'priority', 'difficulty']
CF matrix shape: (100, 4)


task_type,Account Activation,KYC Verification,M-PESA Onboarding,SIM Registration
agent_id,,,,
A001,NaN,3.5900,NaN,NaN
A002,3.230000,3.3725,NaN,NaN
A003,4.530000,4.5040,NaN,4.624
A004,4.782222,NaN,NaN,NaN
A005,4.890000,NaN,NaN,NaN


In [11]:
cf_similarity_matrix = cosine_similarity(
    cf_rating_matrix_filled
)

cf_agent_ids = (
    cf_rating_matrix.index.tolist()
)

cf_agent_index = {
    agent_id: index
    for index, agent_id
    in enumerate(cf_agent_ids)
}


# Average rating for each task type
cf_task_type_means = (
    train_cf_df
    .groupby("task_type")[
        "performance_rating"
    ]
    .mean()
    .to_dict()
)

cf_global_mean = (
    train_cf_df[
        "performance_rating"
    ].mean()
)

print(
    "Similarity matrix shape:",
    cf_similarity_matrix.shape
)

Similarity matrix shape: (100, 100)


In [12]:
def calculate_cf_score(
    agent_id,
    task_id,
    n_neighbors=10
):

    task_type = tasks_lookup[
        task_id
    ]["task_type"]

    agent_position = cf_agent_index[
        agent_id
    ]

    similarities = (
        cf_similarity_matrix[
            agent_position
        ].copy()
    )

    # Do not use the agent itself
    similarities[
        agent_position
    ] = 0.0


    # Sort most similar agents first
    neighbor_positions = np.argsort(
        similarities
    )[::-1]


    ratings = []
    weights = []


    for position in neighbor_positions:

        similarity = similarities[
            position
        ]

        if similarity <= 0:
            continue

        neighbor_id = (
            cf_agent_ids[position]
        )

        if (
            task_type
            not in cf_rating_matrix_raw.columns
        ):
            continue

        neighbor_rating = (
            cf_rating_matrix_raw.loc[
                neighbor_id,
                task_type
            ]
        )

        if pd.notna(
            neighbor_rating
        ):

            ratings.append(
                float(neighbor_rating)
            )

            weights.append(
                float(similarity)
            )

        if len(ratings) >= n_neighbors:
            break


    # Weighted prediction
    if ratings and sum(weights) > 0:

        predicted_rating = np.average(
            ratings,
            weights=weights
        )

    else:

        predicted_rating = (
            cf_task_type_means.get(
                task_type,
                cf_global_mean
            )
        )


    # Ratings are on a 1–5 scale.
    # Normalize to 0–1 for the hybrid model.
    cf_score = (
        float(predicted_rating)
        / 5.0
    )

    cf_score = float(
        np.clip(
            cf_score,
            0.0,
            1.0
        )
    )


    return {
        "cf_score": cf_score,
        "predicted_rating":
            float(predicted_rating),
        "neighbors_used":
            len(ratings)
    }

In [13]:
calculate_cf_score(
    sample_agent_id,
    sample_task_id
)

{'cf_score': 0.833606866483783,
 'predicted_rating': 4.168034332418915,
 'neighbors_used': 10}

### 2.3 Knowledge Graph Component

The Knowledge Graph component represents relationships between agents, tasks, skills and locations.

Historical completion relationships are constructed using training data only.

The Knowledge Graph score considers:

- Agent-to-skill relationships
- Task-to-required-skill relationships
- Shared location relationships
- Previous completion of the same task type
- Historical performance on similar tasks

The resulting score is normalized between 0 and 1 for use in the hybrid recommender.

In [14]:
G = nx.DiGraph()


# --------------------------------------------------
# Agent nodes
# --------------------------------------------------

for _, row in agents_df.iterrows():

    G.add_node(
        f"agent:{row['agent_id']}",
        node_type="Agent",
        agent_id=row["agent_id"],
        name=row["name"],
        specialization=row["specialization"],
        location=row["location"],
        availability=row["availability"],
        workload=row["workload"],
        performance_score=row["performance_score"],
        experience_years=row["experience_years"]
    )


# --------------------------------------------------
# Skill nodes
# --------------------------------------------------

for _, row in skills_df.iterrows():

    G.add_node(
        f"skill:{row['skill_id']}",
        node_type="Skill",
        skill_id=row["skill_id"],
        skill_name=row["skill_name"]
    )


# --------------------------------------------------
# Task nodes
# --------------------------------------------------

for _, row in tasks_df.iterrows():

    G.add_node(
        f"task:{row['task_id']}",
        node_type="Task",
        task_id=row["task_id"],
        task_type=row["task_type"],
        location=row["location"],
        priority=row["priority"],
        difficulty=row["difficulty"],
        status=row["status"]
    )


# --------------------------------------------------
# Location nodes
# --------------------------------------------------

locations = set(
    agents_df["location"].dropna()
).union(
    set(tasks_df["location"].dropna())
)

for location in locations:

    G.add_node(
        f"location:{location}",
        node_type="Location",
        location_name=location
    )


# --------------------------------------------------
# Agent -> Skill
# --------------------------------------------------

for _, row in agent_skills_df.iterrows():

    G.add_edge(
        f"agent:{row['agent_id']}",
        f"skill:{row['skill_id']}",
        relation="HAS_SKILL",
        proficiency_level=row[
            "proficiency_level"
        ]
    )


# --------------------------------------------------
# Task -> Skill
# --------------------------------------------------

for _, row in task_skills_df.iterrows():

    G.add_edge(
        f"task:{row['task_id']}",
        f"skill:{row['skill_id']}",
        relation="REQUIRES_SKILL",
        required_level=row[
            "required_level"
        ]
    )


# --------------------------------------------------
# Agent -> Location
# --------------------------------------------------

for _, row in agents_df.iterrows():

    G.add_edge(
        f"agent:{row['agent_id']}",
        f"location:{row['location']}",
        relation="LOCATED_IN"
    )


# --------------------------------------------------
# Task -> Location
# --------------------------------------------------

for _, row in tasks_df.iterrows():

    G.add_edge(
        f"task:{row['task_id']}",
        f"location:{row['location']}",
        relation="LOCATED_IN"
    )


# --------------------------------------------------
# Historical Agent -> Task
# TRAINING DATA ONLY
# --------------------------------------------------

for _, row in train_df.iterrows():

    G.add_edge(
        f"agent:{row['agent_id']}",
        f"task:{row['task_id']}",
        relation="COMPLETED_TASK",
        performance_rating=row[
            "performance_rating"
        ]
    )


print("KG nodes:", G.number_of_nodes())
print("KG edges:", G.number_of_edges())

KG nodes: 2114
KG edges: 7197


In [15]:
KG_WEIGHTS = {
    "skill_relation": 0.50,
    "location_relation": 0.15,
    "historical_task_type": 0.20,
    "historical_performance": 0.15
}

print(
    "KG weight total:",
    sum(KG_WEIGHTS.values())
)

KG weight total: 1.0


In [16]:
def calculate_kg_features(
    agent_id,
    task_id
):

    agent_node = f"agent:{agent_id}"
    task_node = f"task:{task_id}"

    task_data = G.nodes[
        task_node
    ]


    # ---------------------------------------
    # Required skill relationships
    # ---------------------------------------

    required_skills = {}

    for neighbor in G.successors(
        task_node
    ):

        edge = G[
            task_node
        ][neighbor]

        if (
            edge.get("relation")
            == "REQUIRES_SKILL"
        ):

            required_skills[
                neighbor
            ] = proficiency_map[
                edge["required_level"]
            ]


    matched_skill_scores = []

    for (
        skill_node,
        required_level
    ) in required_skills.items():

        if G.has_edge(
            agent_node,
            skill_node
        ):

            edge = G[
                agent_node
            ][skill_node]

            agent_level = (
                proficiency_map[
                    edge[
                        "proficiency_level"
                    ]
                ]
            )

            matched_skill_scores.append(
                min(
                    agent_level
                    / required_level,
                    1.0
                )
            )

        else:
            matched_skill_scores.append(
                0.0
            )


    if matched_skill_scores:

        skill_relation_score = (
            np.mean(
                matched_skill_scores
            )
        )

    else:
        skill_relation_score = 0.0


    # ---------------------------------------
    # Location relationship
    # ---------------------------------------

    agent_locations = {
        node
        for node in G.successors(
            agent_node
        )
        if G[
            agent_node
        ][node].get("relation")
        == "LOCATED_IN"
    }

    task_locations = {
        node
        for node in G.successors(
            task_node
        )
        if G[
            task_node
        ][node].get("relation")
        == "LOCATED_IN"
    }

    location_relation_score = (
        1.0
        if agent_locations
        & task_locations
        else 0.0
    )


    # ---------------------------------------
    # Historical same task-type experience
    # ---------------------------------------

    target_task_type = (
        task_data["task_type"]
    )

    historical_ratings = []

    for completed_task in G.successors(
        agent_node
    ):

        edge = G[
            agent_node
        ][completed_task]

        if (
            edge.get("relation")
            == "COMPLETED_TASK"
        ):

            if (
                G.nodes[
                    completed_task
                ].get("task_type")
                == target_task_type
            ):

                historical_ratings.append(
                    float(
                        edge[
                            "performance_rating"
                        ]
                    )
                )


    historical_task_type_score = (
        1.0
        if historical_ratings
        else 0.0
    )


    if historical_ratings:

        historical_performance_score = (
            np.mean(
                historical_ratings
            ) / 5.0
        )

    else:
        historical_performance_score = 0.0


    return {
        "skill_relation_score":
            float(skill_relation_score),

        "location_relation_score":
            float(location_relation_score),

        "historical_task_type_score":
            float(
                historical_task_type_score
            ),

        "historical_performance_score":
            float(
                historical_performance_score
            ),

        "historical_matches":
            len(historical_ratings)
    }

In [17]:
def calculate_kg_score(
    agent_id,
    task_id
):

    features = calculate_kg_features(
        agent_id,
        task_id
    )

    kg_score = (
        KG_WEIGHTS["skill_relation"]
        * features[
            "skill_relation_score"
        ]

        + KG_WEIGHTS["location_relation"]
        * features[
            "location_relation_score"
        ]

        + KG_WEIGHTS[
            "historical_task_type"
        ]
        * features[
            "historical_task_type_score"
        ]

        + KG_WEIGHTS[
            "historical_performance"
        ]
        * features[
            "historical_performance_score"
        ]
    )

    return {
        "kg_score": float(kg_score),
        **features
    }

In [18]:
calculate_kg_score(
    sample_agent_id,
    sample_task_id
)

{'kg_score': 0.9957,
 'skill_relation_score': 1.0,
 'location_relation_score': 1.0,
 'historical_task_type_score': 1.0,
 'historical_performance_score': 0.9713333333333333,
 'historical_matches': 12}

## 3. Hybrid Recommendation Model

The hybrid recommender combines the normalized scores produced by the Content-Based, Collaborative Filtering and Knowledge Graph components.

Rather than selecting arbitrary final weights, several weight combinations will be evaluated using the validation dataset.

The validation dataset is used for weight selection only. The test dataset remains untouched until final evaluation.

In [19]:
INITIAL_HYBRID_WEIGHTS = {
    "cbf": 0.35,
    "cf": 0.15,
    "kg": 0.50
}

print(
    "Hybrid weight total:",
    sum(INITIAL_HYBRID_WEIGHTS.values())
)

Hybrid weight total: 1.0


In [20]:
def calculate_hybrid_score(
    agent_id,
    task_id,
    weights=INITIAL_HYBRID_WEIGHTS
):

    cbf_result = calculate_cbf_score(
        agent_id,
        task_id
    )

    cf_result = calculate_cf_score(
        agent_id,
        task_id
    )

    kg_result = calculate_kg_score(
        agent_id,
        task_id
    )

    hybrid_score = (
        weights["cbf"]
        * cbf_result["cbf_score"]

        + weights["cf"]
        * cf_result["cf_score"]

        + weights["kg"]
        * kg_result["kg_score"]
    )

    return {
        "agent_id": agent_id,

        "hybrid_score":
            float(hybrid_score),

        "cbf_score":
            float(
                cbf_result["cbf_score"]
            ),

        "cf_score":
            float(
                cf_result["cf_score"]
            ),

        "kg_score":
            float(
                kg_result["kg_score"]
            )
    }

In [21]:
calculate_hybrid_score(
    sample_agent_id,
    sample_task_id
)

{'agent_id': 'A058',
 'hybrid_score': 0.8665470299725675,
 'cbf_score': 0.6961600000000001,
 'cf_score': 0.833606866483783,
 'kg_score': 0.9957}

In [22]:
MINIMUM_SUITABILITY_SCORE = 0.60


def recommend_agents_hybrid(
    task_id,
    top_k=5,
    weights=INITIAL_HYBRID_WEIGHTS,
    enforce_availability=True
):

    recommendations = []

    for agent_id in agents_df[
        "agent_id"
    ]:

        agent = agents_lookup[
            agent_id
        ]

        # Apply current availability only
        # for operational/live recommendations
        if (
            enforce_availability
            and not bool(
                agent["availability"]
            )
        ):
            continue

        result = calculate_hybrid_score(
            agent_id,
            task_id,
            weights=weights
        )

        recommendations.append(
            result
        )


    recommendations_df = pd.DataFrame(
        recommendations
    )

    recommendations_df = (
        recommendations_df
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_k)
        .reset_index(drop=True)
    )

    recommendations_df.insert(
        0,
        "rank",
        range(
            1,
            len(recommendations_df) + 1
        )
    )


    recommendations_df[
        "decision"
    ] = np.where(
        recommendations_df[
            "hybrid_score"
        ]
        >= MINIMUM_SUITABILITY_SCORE,
        "Suitable",
        "Supervisor review required"
    )

    return recommendations_df

In [23]:
sample_hybrid_task_id = (
    tasks_df[
        tasks_df["status"] == "Pending"
    ]
    .iloc[0]["task_id"]
)

print(
    "Task:",
    sample_hybrid_task_id
)

recommend_agents_hybrid(
    sample_hybrid_task_id,
    top_k=5
)

Task: T004


,rank,agent_id,hybrid_score,cbf_score,cf_score,kg_score,decision
0,1,A017,0.863569,0.71663,0.826555,0.97753,Suitable
1,2,A043,0.837623,0.88337,0.768124,0.82645,Suitable
2,3,A040,0.816308,0.80436,0.836683,0.81856,Suitable
3,4,A049,0.727156,0.66359,0.883665,0.72470,Suitable
4,5,A059,0.726417,0.68219,0.827001,0.72720,Suitable


## 4. Hybrid Weight Selection

The initial hybrid weights were selected only as a starting point.

To select more defensible weights, multiple combinations of Content-Based, Collaborative Filtering and Knowledge Graph weights are evaluated using the validation dataset.

Each component is required to contribute at least 10% to ensure that the resulting model remains a true hybrid recommender.

The primary selection metric is Hit Rate@5, with MAP@5 used to distinguish between configurations with similar Top-5 performance.

The test dataset is not used during weight selection.

In [24]:
# Precompute component scores for every
# validation task and every agent

component_score_records = []

validation_task_ids = (
    validation_df["task_id"]
    .unique()
)

for task_id in validation_task_ids:

    for agent_id in agents_df["agent_id"]:

        cbf_result = calculate_cbf_score(
            agent_id,
            task_id
        )

        cf_result = calculate_cf_score(
            agent_id,
            task_id
        )

        kg_result = calculate_kg_score(
            agent_id,
            task_id
        )

        component_score_records.append({
            "task_id": task_id,
            "agent_id": agent_id,
            "cbf_score":
                cbf_result["cbf_score"],
            "cf_score":
                cf_result["cf_score"],
            "kg_score":
                kg_result["kg_score"]
        })


component_scores_df = pd.DataFrame(
    component_score_records
)

print(
    "Precomputed rows:",
    len(component_scores_df)
)

display(
    component_scores_df.head()
)

Precomputed rows: 19600


,task_id,agent_id,cbf_score,cf_score,kg_score
0,T1892,A001,0.25518,0.787874,0.150000
1,T1892,A002,0.63250,0.829578,0.796900
2,T1892,A003,0.57555,0.832631,0.835900
3,T1892,A004,0.65285,0.855578,0.843467
4,T1892,A005,0.85954,0.853168,0.846700


In [25]:
def evaluate_hybrid_weights(
    weights,
    k=5
):

    scored = component_scores_df.copy()

    scored["hybrid_score"] = (
        weights["cbf"]
        * scored["cbf_score"]

        + weights["cf"]
        * scored["cf_score"]

        + weights["kg"]
        * scored["kg_score"]
    )


    hits = []
    precisions = []
    recalls = []
    average_precisions = []


    for _, row in validation_df.iterrows():

        task_id = row["task_id"]
        relevant_agent = row["agent_id"]

        task_ranking = (
            scored[
                scored["task_id"]
                == task_id
            ]
            .sort_values(
                "hybrid_score",
                ascending=False
            )
            .head(k)
        )

        recommended_agents = (
            task_ranking[
                "agent_id"
            ].tolist()
        )


        if relevant_agent in recommended_agents:

            rank = (
                recommended_agents.index(
                    relevant_agent
                ) + 1
            )

            hits.append(1)
            precisions.append(1.0 / k)
            recalls.append(1.0)
            average_precisions.append(
                1.0 / rank
            )

        else:

            hits.append(0)
            precisions.append(0.0)
            recalls.append(0.0)
            average_precisions.append(0.0)


    return {
        "Precision@K":
            np.mean(precisions),

        "Recall@K":
            np.mean(recalls),

        "Hit Rate@K":
            np.mean(hits),

        "MAP@K":
            np.mean(
                average_precisions
            )
    }

In [26]:
weight_search_results = []

weight_values = [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8
]


for cbf_weight in weight_values:

    for cf_weight in weight_values:

        kg_weight = round(
            1.0
            - cbf_weight
            - cf_weight,
            1
        )

        # All three components must contribute
        if kg_weight < 0.1:
            continue

        if kg_weight > 0.8:
            continue


        weights = {
            "cbf": cbf_weight,
            "cf": cf_weight,
            "kg": kg_weight
        }


        metrics = evaluate_hybrid_weights(
            weights,
            k=5
        )


        weight_search_results.append({
            "cbf_weight":
                cbf_weight,

            "cf_weight":
                cf_weight,

            "kg_weight":
                kg_weight,

            "Hit Rate@5":
                metrics[
                    "Hit Rate@K"
                ],

            "MAP@5":
                metrics[
                    "MAP@K"
                ]
        })


weight_search_df = pd.DataFrame(
    weight_search_results
)

In [27]:
weight_search_df = (
    weight_search_df
    .sort_values(
        [
            "Hit Rate@5",
            "MAP@5"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

weight_search_df.head(10)

,cbf_weight,cf_weight,kg_weight,Hit Rate@5,MAP@5
0,0.3,0.3,0.4,0.336735,0.180697
1,0.2,0.3,0.5,0.336735,0.179507
2,0.2,0.1,0.7,0.336735,0.178146
3,0.2,0.2,0.6,0.336735,0.178146
4,0.2,0.4,0.4,0.331633,0.182313
5,0.1,0.3,0.6,0.331633,0.171259
6,0.3,0.1,0.6,0.326531,0.175000
7,0.1,0.1,0.8,0.326531,0.173554
8,0.3,0.2,0.5,0.326531,0.171429
9,0.1,0.6,0.3,0.326531,0.164711


### Selected Hybrid Weights

The selected hybrid configuration assigns:

- 30% to Content-Based Filtering
- 30% to Collaborative Filtering
- 40% to Knowledge Graph reasoning

Several predefined weight combinations were evaluated using the validation dataset.

Multiple configurations achieved the same highest Hit Rate@5 of approximately 33.67%. The 30/30/40 configuration was selected because it achieved the highest MAP@5 among the tied configurations.

The selected weights should therefore be interpreted as the best configuration within the tested validation grid rather than a globally optimal set of weights.

The test dataset was not used during weight selection.

In [28]:
SELECTED_HYBRID_WEIGHTS = {
    "cbf": 0.30,
    "cf": 0.30,
    "kg": 0.40
}

print(
    "Selected weights:",
    SELECTED_HYBRID_WEIGHTS
)

print(
    "Total:",
    sum(
        SELECTED_HYBRID_WEIGHTS.values()
    )
)

Selected weights: {'cbf': 0.3, 'cf': 0.3, 'kg': 0.4}
Total: 1.0


In [29]:
hybrid_evaluation_summary = []

for k in [1, 3, 5]:

    metrics = evaluate_hybrid_weights(
        SELECTED_HYBRID_WEIGHTS,
        k=k
    )

    hybrid_evaluation_summary.append({
        "K": k,
        "Precision@K":
            metrics["Precision@K"],
        "Recall@K":
            metrics["Recall@K"],
        "Hit Rate@K":
            metrics["Hit Rate@K"],
        "MAP@K":
            metrics["MAP@K"]
    })


hybrid_evaluation_summary_df = pd.DataFrame(
    hybrid_evaluation_summary
)

hybrid_evaluation_summary_df

,K,Precision@K,Recall@K,Hit Rate@K,MAP@K
0,1,0.102041,0.102041,0.102041,0.102041
1,3,0.076531,0.229592,0.229592,0.156463
2,5,0.067347,0.336735,0.336735,0.180697


## 5. Comparison of All Recommendation Models

The selected hybrid recommender is compared with the three individual recommendation approaches using the same validation dataset.

The comparison considers Hit Rate and Mean Average Precision at different recommendation list sizes.

This allows the contribution of the hybrid approach to be assessed without using the reserved test dataset.

In [30]:
model_comparison_df = pd.DataFrame({
    "K": hybrid_evaluation_summary_df["K"],

    "Content-Based Hit Rate":
        cbf_metrics["Hit Rate@K"],

    "Collaborative Filtering Hit Rate":
        cf_metrics["Hit Rate@K"],

    "Knowledge Graph Hit Rate":
        kg_metrics["Hit Rate@K"],

    "Hybrid Hit Rate":
        hybrid_evaluation_summary_df["Hit Rate@K"],

    "Content-Based MAP":
        cbf_metrics["MAP@K"],

    "Collaborative Filtering MAP":
        cf_metrics["MAP@K"],

    "Knowledge Graph MAP":
        kg_metrics["MAP@K"],

    "Hybrid MAP":
        hybrid_evaluation_summary_df["MAP@K"]
})

model_comparison_df

,K,Content-Based Hit Rate,Collaborative Filtering Hit Rate,Knowledge Graph Hit Rate,Hybrid Hit Rate,Content-Based MAP,Collaborative Filtering MAP,Knowledge Graph MAP,Hybrid MAP
0,1,0.091837,0.015306,0.091837,0.102041,0.091837,0.015306,0.091837,0.102041
1,3,0.209184,0.051020,0.229592,0.229592,0.138605,0.030612,0.152211,0.156463
2,5,0.255102,0.091837,0.341837,0.336735,0.149575,0.040051,0.176956,0.180697


### Interpretation

The Hybrid Recommender achieved the strongest Top-1 performance, with a Hit Rate@1 of approximately 10.20%.

At Top-3, the Hybrid and Knowledge Graph approaches achieved the same Hit Rate of approximately 22.96%, while the Hybrid achieved a slightly higher MAP@3.

At Top-5, the Knowledge Graph achieved the highest Hit Rate at approximately 34.18%, compared with approximately 33.67% for the Hybrid. However, the Hybrid achieved a slightly higher MAP@5 of approximately 18.07%, compared with approximately 17.70% for the Knowledge Graph.

This indicates that the hybrid approach tends to rank relevant agents slightly higher, while the standalone Knowledge Graph provides slightly better overall Top-5 coverage.

The results therefore do not suggest that the Hybrid model dominates every individual model. Instead, the Hybrid model provides a balanced combination of content information, collaborative behaviour and graph-based relationships.

These results are based on synthetic development data and are not interpreted as final real-world system performance.

## 6. Explainable Hybrid Recommendations

The final recommendation should not only provide a score but also explain the main reasons why an agent was recommended.

The explanation combines evidence from Content-Based Filtering, Collaborative Filtering and Knowledge Graph reasoning.

In [31]:
def generate_hybrid_explanation(
    agent_id,
    task_id
):

    cbf_result = calculate_cbf_score(
        agent_id,
        task_id
    )

    cf_result = calculate_cf_score(
        agent_id,
        task_id
    )

    kg_result = calculate_kg_score(
        agent_id,
        task_id
    )

    reasons = []


    # Content-Based evidence
    if cbf_result["skill_match"] == 1.0:
        reasons.append(
            "fully matches the required skills"
        )

    elif cbf_result["skill_match"] >= 0.5:
        reasons.append(
            "matches some of the required skills"
        )


    if cbf_result["specialization_match"] == 1.0:
        reasons.append(
            "has the required specialization"
        )


    if cbf_result["location_match"] == 1.0:
        reasons.append(
            "is in the same location as the task"
        )


    # Collaborative evidence
    if cf_result["cf_score"] >= 0.8:
        reasons.append(
            "has strong collaborative performance evidence"
        )


    # Knowledge Graph evidence
    if kg_result["historical_task_type_score"] == 1.0:
        reasons.append(
            "has previously completed the same type of task"
        )


    if kg_result["historical_performance_score"] >= 0.8:
        reasons.append(
            "has strong historical performance on similar tasks"
        )


    if not reasons:
        return (
            "Limited supporting evidence is available; "
            "supervisor review is recommended."
        )


    return "; ".join(reasons) + "."

In [32]:
final_sample_recommendations = (
    recommend_agents_hybrid(
        sample_hybrid_task_id,
        top_k=5,
        weights=SELECTED_HYBRID_WEIGHTS
    )
)

final_sample_recommendations[
    "explanation"
] = final_sample_recommendations.apply(
    lambda row:
        generate_hybrid_explanation(
            row["agent_id"],
            sample_hybrid_task_id
        ),
    axis=1
)

final_sample_recommendations[
    [
        "rank",
        "agent_id",
        "hybrid_score",
        "decision",
        "explanation"
    ]
]

,rank,agent_id,hybrid_score,decision,explanation
0,1,A017,0.853968,Suitable,fully matches the required skills; is in the s...
1,2,A043,0.826028,Suitable,fully matches the required skills; has the req...
2,3,A040,0.819737,Suitable,fully matches the required skills; has the req...
3,4,A080,0.757199,Suitable,matches some of the required skills; has the r...
4,5,A049,0.754057,Suitable,matches some of the required skills; has the r...


## 7. Save Hybrid Experiment Evidence

The selected hybrid configuration, validation metrics, model comparison, weight-search results and Top-5 validation predictions are saved to support reproducibility and Chapter 5 documentation.

In [33]:
hybrid_experiment_path = (
    experiments_path
    / "HYBRID_RUN_001"
)

hybrid_experiment_path.mkdir(
    parents=True,
    exist_ok=True
)


hybrid_config = {
    "run_id": "HYBRID_RUN_001",
    "model": "Hybrid Recommender",
    "dataset_version": "synthetic_v1",
    "random_seed": 42,

    "components": [
        "Content-Based Filtering",
        "Collaborative Filtering",
        "Knowledge Graph"
    ],

    "selected_weights": {
        "cbf": 0.30,
        "cf": 0.30,
        "kg": 0.40
    },

    "minimum_suitability_score": 0.60,

    "weight_selection_metric": [
        "Hit Rate@5",
        "MAP@5"
    ],

    "training_split": 0.70,
    "validation_split": 0.15,
    "test_split": 0.15,

    "evaluation_data": "validation",
    "test_data_used": False
}


with open(
    hybrid_experiment_path / "config.json",
    "w"
) as file:

    json.dump(
        hybrid_config,
        file,
        indent=4
    )


hybrid_evaluation_summary_df.to_csv(
    hybrid_experiment_path
    / "validation_metrics.csv",
    index=False
)


model_comparison_df.to_csv(
    hybrid_experiment_path
    / "model_comparison.csv",
    index=False
)


weight_search_df.to_csv(
    hybrid_experiment_path
    / "weight_search.csv",
    index=False
)


print(
    "Basic hybrid experiment evidence saved."
)

Basic hybrid experiment evidence saved.


In [34]:
# Calculate selected hybrid score once
selected_scores_df = component_scores_df.copy()

selected_scores_df[
    "hybrid_score"
] = (
    SELECTED_HYBRID_WEIGHTS["cbf"]
    * selected_scores_df["cbf_score"]

    + SELECTED_HYBRID_WEIGHTS["cf"]
    * selected_scores_df["cf_score"]

    + SELECTED_HYBRID_WEIGHTS["kg"]
    * selected_scores_df["kg_score"]
)


hybrid_prediction_records = []


for _, row in validation_df.iterrows():

    task_id = row["task_id"]
    relevant_agent = row["agent_id"]

    top5 = (
        selected_scores_df[
            selected_scores_df["task_id"]
            == task_id
        ]
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(5)
        .reset_index(drop=True)
    )


    for index, prediction in top5.iterrows():

        hybrid_prediction_records.append({
            "task_id":
                task_id,

            "relevant_agent_id":
                relevant_agent,

            "recommended_agent_id":
                prediction["agent_id"],

            "rank":
                index + 1,

            "hybrid_score":
                prediction["hybrid_score"],

            "cbf_score":
                prediction["cbf_score"],

            "cf_score":
                prediction["cf_score"],

            "kg_score":
                prediction["kg_score"],

            "is_relevant":
                int(
                    prediction["agent_id"]
                    == relevant_agent
                )
        })


hybrid_predictions_k5_df = pd.DataFrame(
    hybrid_prediction_records
)


hybrid_predictions_k5_df.to_csv(
    hybrid_experiment_path
    / "validation_predictions_k5.csv",
    index=False
)


print(
    "Hybrid validation predictions saved."
)

for file in hybrid_experiment_path.iterdir():
    print("-", file.name)

Hybrid validation predictions saved.
- config.json
- model_comparison.csv
- validation_metrics.csv
- validation_predictions_k5.csv
- weight_search.csv


## Conclusion

The Hybrid Recommender was successfully developed by combining Content-Based Filtering, Collaborative Filtering and Knowledge Graph reasoning.

Validation-based weight selection identified a configuration of 30% Content-Based Filtering, 30% Collaborative Filtering and 40% Knowledge Graph reasoning.

The selected Hybrid model achieved:

- Hit Rate@1 of approximately 10.20%
- Hit Rate@3 of approximately 22.96%
- Hit Rate@5 of approximately 33.67%
- MAP@5 of approximately 18.07%

The Hybrid model achieved the strongest Top-1 performance among the evaluated models and slightly improved ranking quality at Top-5 compared with the standalone Knowledge Graph model.

The standalone Knowledge Graph maintained a slightly higher Hit Rate@5 of approximately 34.18%, showing that the Hybrid model does not outperform every individual model on every metric.

The Hybrid approach nevertheless provides a balanced recommendation strategy by combining agent-task compatibility, collaborative historical behaviour and graph-based relationships.

Explainable recommendations were also generated so that supervisors can understand the main reasons supporting each suggested agent.

The results use synthetic development data and therefore represent development evidence rather than final real-world performance.

The test dataset remains unused and reserved for final evaluation.